# 02 — Spark Structured Streaming → Iceberg

**Pre-requisite:** Start the event producer in a separate terminal first:
```bash
uv run python producer.py
```

**What this notebook does:**
1. Creates Iceberg table `iceberg_catalog.my_database.user_events`
2. Reads a stream of JSON files from `./app/data/streaming_input/`
3. Writes every micro-batch into the Iceberg table (append mode)
4. Lets you monitor row count in real time
5. Stops the stream cleanly

> The producer writes a new 2-row JSONL file every 10 s.  
> Spark triggers every 15 s so you'll see rows accumulate in batches.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType
import os

spark = SparkSession.builder.appName("Spark-Developer").master("local[*]").getOrCreate()

spark

## Step 1 — Create Iceberg target table

In [2]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS iceberg_catalog.my_database.user_events (
        event_id   STRING,
        user_id    STRING,
        event_type STRING,
        page       STRING,
        timestamp  STRING,
        session_id STRING
    )
    USING iceberg
    TBLPROPERTIES ('format-version' = '2')
""")

print("Table ready: iceberg_catalog.my_database.user_events")
spark.sql("DESCRIBE iceberg_catalog.my_database.user_events").show()

Table ready: iceberg_catalog.my_database.user_events
+----------+---------+-------+
|  col_name|data_type|comment|
+----------+---------+-------+
|  event_id|   string|   NULL|
|   user_id|   string|   NULL|
|event_type|   string|   NULL|
|      page|   string|   NULL|
| timestamp|   string|   NULL|
|session_id|   string|   NULL|
+----------+---------+-------+



## Step 2 — Define schema and streaming reader

In [ ]:
STREAMING_INPUT = "./app/data/streaming_input"
CHECKPOINT_DIR  = "./.tmp/checkpoint_user_events"

os.makedirs(STREAMING_INPUT, exist_ok=True)
os.makedirs(CHECKPOINT_DIR,  exist_ok=True)

event_schema = StructType([
    StructField("event_id",   StringType(), True),
    StructField("user_id",    StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("page",       StringType(), True),
    StructField("timestamp",  StringType(), True),
    StructField("session_id", StringType(), True),
])

stream_df = (
    spark.readStream
         .format("json")
         .schema(event_schema)
         .option("maxFilesPerTrigger", 10)
         .load(STREAMING_INPUT)
)

print("Is streaming:", stream_df.isStreaming)
stream_df.printSchema()

## Step 3 — Start streaming write to Iceberg

> **Note:** this cell returns immediately. The stream runs in the background.  
> Use the monitoring cell below to observe progress.

In [4]:
query = stream_df.writeStream \
     .format("iceberg") \
     .outputMode("append") \
     .option("checkpointLocation", CHECKPOINT_DIR) \
     .trigger(processingTime="15 seconds") \
     .toTable("iceberg_catalog.my_database.user_events")

print(f"Stream started  — id: {query.id}")
print(f"Name           : {query.name}")
print(f"Status         : {query.status}")

Stream started  — id: 306a6a5a-22ec-4ae1-b1ec-e2a25c5bcf43
Name           : None
Status         : {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


26/04/25 14:28:03 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


## Step 4 — Monitor (re-run this cell repeatedly)

Re-run this cell every ~15–30 s to see new rows arriving.

In [5]:
print("Active :", query.isActive)
print("Status :", query.status)

if query.lastProgress:
    prog = query.lastProgress
    print(f"Batch  : {prog['batchId']}")
    print(f"Rows/s : {prog['processedRowsPerSecond']}")
    print(f"Input  : {prog['numInputRows']} rows in last batch")

print()
spark.sql("SELECT COUNT(*) AS total_events FROM iceberg_catalog.my_database.user_events").show()
spark.sql("""
    SELECT event_type, COUNT(*) AS cnt
    FROM iceberg_catalog.my_database.user_events
    GROUP BY event_type
    ORDER BY cnt DESC
""").show()

Active : True
Status : {'message': 'Waiting for next trigger', 'isDataAvailable': False, 'isTriggerActive': False}
Batch  : 45
Rows/s : 0.0
Input  : 0 rows in last batch



+------------+
|total_events|
+------------+
|         174|
+------------+

+-----------+---+
| event_type|cnt|
+-----------+---+
|   purchase| 38|
|     search| 34|
|     logout| 32|
|add_to_cart| 28|
|      click| 26|
|  page_view| 16|
+-----------+---+



## Step 5 — Stop the stream

In [6]:
query.stop()
print("Stream stopped.")

# Final row count
total = spark.sql("SELECT COUNT(*) AS total FROM iceberg_catalog.my_database.user_events").collect()[0]["total"]
print(f"Total events ingested: {total}")

Stream stopped.
Total events ingested: 174


26/04/25 14:28:29 WARN DAGScheduler: Failed to cancel job group d330cd63-c72c-482a-9fd3-508201cf70bd. Cannot find active jobs for it.
26/04/25 14:28:29 WARN DAGScheduler: Failed to cancel job group d330cd63-c72c-482a-9fd3-508201cf70bd. Cannot find active jobs for it.
